# Gravitational Averaging Visualization

This notebook demonstrates how gravitational averaging creates a balance in mass distribution.

## Key Concepts
- **Nodes** represent masses (e.g., stars, particles)
- **Edges** represent gravitational connections based on force strength
- **Filament Structure** naturally emerges through gravitational averaging

## Parameters
- `num_nodes`: Number of masses in the simulation
- `neighbors`: Number of strongest gravitational connections per node

## Customize the Model
Try changing `num_nodes` or `neighbors` to observe different behaviors.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

# Gravitational Force Calculation

def gravitational_force(m1, m2, r, G=6.67430e-11):
    """Calculate the gravitational force between two masses."""
    return G * (m1 * m2) / (r ** 2 + 1e-5)  # Avoid division by zero

# Create a Gravitational Averaging Graph

def create_gravitational_averaging_graph(num_nodes=50, neighbors=5):
    G = nx.Graph()
    positions = {}

    # Randomly distribute nodes with varying masses
    for i in range(num_nodes):
        mass = np.random.uniform(1e6, 1e9)  # Random mass for each node
        pos_x, pos_y = np.random.uniform(-25, 25, 2)

        G.add_node(i, mass=mass)
        positions[i] = np.array([pos_x, pos_y])

    # Connect nodes to their 'neighbors' strongest connections
    for i in range(num_nodes):
        distances = []
        for j in range(num_nodes):
            if i != j:
                distance = np.linalg.norm(positions[i] - positions[j])
                force = gravitational_force(G.nodes[i]['mass'], G.nodes[j]['mass'], distance)
                distances.append((j, force))

        # Sort and connect to the strongest neighbors
        distances.sort(key=lambda x: -x[1])
        for j, _ in distances[:neighbors]:
            if not G.has_edge(i, j):
                G.add_edge(i, j, weight=force)

    return G, positions

# Plot the Gravitational Averaging Graph

def plot_gravitational_averaging_graph(G, positions):
    plt.figure(figsize=(12, 12))

    masses = [G.nodes[n]['mass'] for n in G.nodes]
    max_mass = max(masses)
    normalized_sizes = [((m / max_mass) * 300) + 50 for m in masses]

    nx.draw_networkx_nodes(G, positions, node_size=normalized_sizes, node_color='purple', alpha=0.7)

    max_force = max([data['weight'] for _, _, data in G.edges(data=True)])
    for (u, v, data) in G.edges(data=True):
        weight = data['weight']
        normalized_weight = (weight / max_force) * 100
        nx.draw_networkx_edges(G, positions, edgelist=[(u, v)], width=normalized_weight, alpha=0.6, edge_color='gray')

    plt.title("Gravitational Averaging Network")
    plt.xlabel("X Position")
    plt.ylabel("Y Position")
    plt.axis('equal')
    plt.grid(True)
    plt.show()

# Example Execution
G, positions = create_gravitational_averaging_graph(num_nodes=100, neighbors=5)
plot_gravitational_averaging_graph(G, positions)
